# Bronze to Silver

Tratamento das tabelas `v_energy.bronze` para `v_energy.silver`, uma seção por tabela. Cada decisão de limpeza
tem um ID (`UC-01`, `FAT-02`...) registrado em `docs/qualidade-dados.md`.

**Como executar**
- `Run All` processa as tabelas em sequência. Com `gravar = nao` (padrão), cada tabela é tratada e **validada**, sem gravar nada.
- Com `gravar = sim`, cada tabela só é gravada (Delta, `overwrite`) depois de passar em todas as validações, e a contagem gravada é conferida.
- Uma tabela que falha não é gravada; a seção **Resumo** falha no fim se alguma tabela falhou ou não foi executada.
- Ordem obrigatória: `unidades_consumidoras` antes de `faturas` e `leituras` (checagem de UC fora do cadastro).

**Regras comuns da Silver**
- Tabelas em `v_energy.silver.<mesmo nome da Bronze>`, com os **mesmos nomes de coluna** da Bronze (renomeação só na Gold).
- Datas como `date`/`timestamp`, valores e kWh como `decimal`, IDs como texto.
- Problema que não dá para corrigir vira coluna `flag_<motivo>` (true/false), sem apagar a linha.
- `source_file` e `ingestion_timestamp` da Bronze são mantidas.
- Nenhum valor preenchido pode virar nulo em silêncio: toda conversão é conferida antes de gravar.

**Confidencial:** as saídas das células têm contagens derivadas do dataset. Limpe os outputs antes de exportar ou commitar.

## 0. Configuração
Execute as duas células desta seção antes de qualquer tabela.

In [0]:
import traceback
from datetime import datetime

from pyspark.sql import Window
from pyspark.sql import functions as F

dbutils.widgets.dropdown("gravar", "nao", ["nao", "sim"], "Gravar na Silver?")
GRAVAR = dbutils.widgets.get("gravar") == "sim"

CATALOG = "v_energy"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
COLS_TECNICAS = ["source_file", "ingestion_timestamp"]
NULOS_DISFARCADOS = ["", "NULL", "N/A", "NA", "-", "NONE", "NAN"]
TABELAS_ESPERADAS = ['unidades_consumidoras', 'faturas', 'leituras', 'medidores', 'geracao_distribuida', 'ocorrencias_rede', 'atendimentos', 'inspecoes']
VALIDACAO_ESPERADA = True  # seção 9 (validação cruzada) faz parte deste notebook

if GRAVAR:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

RESULTADOS = {}  # status de cada tabela, usado no Resumo
SILVER_DF = {}  # DataFrames tratados nesta execução (as tabelas seguintes usam daqui)
print(f"Bronze: {BRONZE} | Silver: {SILVER} | gravar = {'sim' if GRAVAR else 'nao'}")

In [0]:
# ---------- Limpeza e conversão ----------

def limpar(c):
    """Tira espaços das pontas e transforma nulos disfarçados ('', NULL, N/A, -...) em nulo."""
    s = F.trim(F.col(c))
    return F.when(F.upper(s).isin(NULOS_DISFARCADOS), None).otherwise(s)


def limpar_todas(df, colunas):
    return df.select(*[limpar(c).alias(c) if c in colunas else F.col(c) for c in df.columns])


def sem_acento(c):
    """Maiúsculas, sem espaços nas pontas e sem acento: usado para comparar e mapear grafias."""
    return F.translate(F.upper(F.trim(c)), "ÁÀÂÃÉÊÍÓÔÕÚÜÇ", "AAAAEEIOOOUUC")


FORMATOS_DATA = [  # formatos confirmados no profiling (o americano pelos valores acima de 12)
    (r"^\d{4}-\d{2}-\d{2}$", "yyyy-MM-dd"),
    (r"^\d{2}/\d{2}/\d{4}$", "dd/MM/yyyy"),
    (r"^\d{2}\.\d{2}\.\d{4}$", "dd.MM.yyyy"),
    (r"^\d{2}-\d{2}-\d{4}$", "MM-dd-yyyy"),
]


def data_multi(c):
    """Converte cada linha pelo seu formato. Formato desconhecido vira nulo e é barrado por exigir_sem_perda."""
    regra = None
    for padrao, formato in FORMATOS_DATA:
        convertida = F.expr(f"CAST(try_to_timestamp(`{c}`, '{formato}') AS DATE)")
        regra = F.when(F.col(c).rlike(padrao), convertida) if regra is None else regra.when(F.col(c).rlike(padrao), convertida)
    return regra


def dec(c, precisao=12, escala=2):
    """Texto para decimal; vírgula decimal vira ponto. try_cast não derruba a célula no modo ANSI."""
    return F.expr(f"try_cast(replace(`{c}`, ',', '.') AS DECIMAL({precisao},{escala}))")


def ts(c, formato="yyyy-MM-dd HH:mm:ss"):
    return F.expr(f"try_to_timestamp(`{c}`, '{formato}')")


# ---------- Travas: se alguma falhar, a tabela não é gravada ----------

def guardar_texto(df, colunas):
    """Guarda o texto original das colunas que serão convertidas, para conferir depois."""
    for c in colunas:
        df = df.withColumn(f"_txt_{c}", F.col(c))
    return df


def exigir_sem_perda(df, colunas):
    """Nenhum valor preenchido pode virar nulo na conversão (formato inesperado)."""
    r = df.select(*[
        F.sum((F.col(f"_txt_{c}").isNotNull() & F.col(c).isNull()).cast("int")).alias(c) for c in colunas
    ]).first().asDict()
    perdidos = {c: n for c, n in r.items() if n}
    assert not perdidos, f"valores preenchidos que viraram nulo na conversão: {perdidos}"
    return df.drop(*[f"_txt_{c}" for c in colunas])


def exigir_zero(df, regras):
    """regras = {nome: condição}. Toda condição precisa ter 0 linhas."""
    r = df.select(*[
        F.sum(F.coalesce(cond, F.lit(False)).cast("int")).alias(nome) for nome, cond in regras.items()
    ]).first().asDict()
    violadas = {nome: n for nome, n in r.items() if n}
    assert not violadas, f"regras violadas (linhas): {violadas}"


def exigir_chave_unica(df, chave):
    total = df.count()
    unicos = df.select(*chave).distinct().count()
    assert total == unicos, f"chave {chave} duplicada: {total - unicos:,} linhas"
    return total


def exigir_colunas(df, esperadas):
    """A Silver seleciona colunas pelo nome: coluna nova ou ausente na Bronze é erro, nunca descarte silencioso."""
    origem = set(df.columns) - set(COLS_TECNICAS)
    assert origem == set(esperadas), f"colunas diferentes do esperado. Sobrando: {origem - set(esperadas)} | Faltando: {set(esperadas) - origem}"


def exigir_sem_orfaos(df, nome):
    orfaos = df.join(obter("unidades_consumidoras").select("id_uc"), "id_uc", "left_anti").count()
    assert orfaos == 0, f"{nome}: {orfaos:,} linhas com id_uc fora do cadastro"


def contar_flags(df):
    flags = [c for c in df.columns if c.startswith("flag_")]
    if not flags:
        return {}
    return df.select(*[F.sum(F.col(c).cast("int")).alias(c) for c in flags]).first().asDict()


# ---------- Execução ----------

def obter(nome):
    """DataFrame tratado nesta execução; se a seção não rodou agora, lê a tabela Silver já gravada."""
    return SILVER_DF[nome] if nome in SILVER_DF else spark.table(f"{SILVER}.{nome}")


def executar(nome, construir):
    """Trata e valida; só grava com gravar = sim e depois de passar em todas as travas."""
    print(f"=== {nome}")
    try:
        df, info = construir()
        SILVER_DF[nome] = df
        if GRAVAR:
            destino = f"{SILVER}.{nome}"
            df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(destino)
            gravadas = spark.table(destino).count()
            assert gravadas == info["linhas"], f"gravadas {gravadas:,} != validadas {info['linhas']:,}"
            SILVER_DF[nome] = spark.table(destino)
            status = "gravada e conferida"
        else:
            status = "validada (sem gravar)"
        RESULTADOS[nome] = {"status": status, **info}
        print(f"[OK] {nome}: {status} | {info}")
    except Exception as e:
        RESULTADOS[nome] = {"status": "FALHA", "erro": f"{type(e).__name__}: {e}"}
        print(f"[FALHA] {nome}: nada foi gravado.")
        traceback.print_exc()

## 1. unidades_consumidoras
`v_energy.bronze.unidades_consumidoras` → `v_energy.silver.unidades_consumidoras`. Regras UC-01 a UC-11 de `docs/qualidade-dados.md`.

**Tratamentos**
- Cabeçalho repetido removido (UC-01); espaços e nulos disfarçados convertidos em nulo (UC-09).
- `nome_titular` em maiúsculas (UC-04); `cpf_cnpj` só com dígitos, como texto (UC-05); `uf` padronizada para `PE`, depois de conferir que todas as grafias são de Pernambuco (UC-06); `classe` com as 5 grafias do log (UC-07).
- Cópias exatas e cópias com formatação diferente removidas depois da padronização (UC-02, UC-03): 1 linha por `id_uc`.
- `data_ligacao` como `date`; `latitude` e `longitude` como `decimal(9,6)` (UC-11: precisão de origem de 2 casas).

**Flag, sem apagar a linha**
- `flag_cep_ausente`: CEP nulo ou sem 8 dígitos; o CEP inválido vira nulo (UC-08).

**Validações antes de gravar:** `id_uc` único, `classe` dentro do mapa, coerência classe × subgrupo, nenhuma conversão perdida.

In [0]:
CLASSES = {"RESIDENCIAL": "Residencial", "COMERCIAL": "Comercial", "INDUSTRIAL": "Industrial",
           "RURAL": "Rural", "PODER PUBLICO": "Poder Público"}
SUBGRUPO_DA_CLASSE = {"Residencial": "B1", "Rural": "B2", "Comercial": "B3", "Poder Público": "B3", "Industrial": "A4"}


def construir_unidades_consumidoras():
    df = spark.table(f"{BRONZE}.unidades_consumidoras")
    origem = [c for c in df.columns if c not in COLS_TECNICAS]

    df = df.filter(F.col("id_uc").isNull() | (F.upper(F.trim("id_uc")) != "ID_UC"))   # UC-01 (chave nula não some aqui)
    df = limpar_todas(df, origem)                                              # UC-09
    exigir_zero(df, {"uf_fora_de_pe": F.col("uf").isNotNull() & ~F.upper("uf").isin("PE", "PERNAMBUCO", "P.E.")})

    mapa_classe = F.create_map(*[F.lit(x) for kv in CLASSES.items() for x in kv])
    df = (df
        .withColumn("nome_titular", F.upper("nome_titular"))                   # UC-04
        .withColumn("cpf_cnpj", F.regexp_replace("cpf_cnpj", r"[^0-9]", ""))   # UC-05 (texto)
        .withColumn("uf", F.when(F.col("uf").isNotNull(), F.lit("PE")))        # UC-06 (nula continua nula)
        .withColumn("classe", mapa_classe[sem_acento("classe")])               # UC-07
        .withColumn("cep", F.regexp_replace("cep", r"[^0-9]", ""))
        .withColumn("cep", F.when(F.length("cep") == 8, F.col("cep")))         # UC-08: inválido vira nulo
        .withColumn("flag_cep_ausente", F.col("cep").isNull()))

    df = guardar_texto(df, ["data_ligacao", "latitude", "longitude"])
    df = (df
        .withColumn("data_ligacao", data_multi("data_ligacao"))
        .withColumn("latitude", dec("latitude", 9, 6))                          # UC-11
        .withColumn("longitude", dec("longitude", 9, 6)))
    df = exigir_sem_perda(df, ["data_ligacao", "latitude", "longitude"])

    df = df.dropDuplicates(origem)                                             # UC-02 e UC-03
    df = df.select(*origem, "flag_cep_ausente", *COLS_TECNICAS)

    exigir_zero(df, {"id_uc_nulo": F.col("id_uc").isNull()})
    total = exigir_chave_unica(df, ["id_uc"])
    mapa_sub = F.create_map(*[F.lit(x) for kv in SUBGRUPO_DA_CLASSE.items() for x in kv])
    exigir_zero(df, {
        "classe_fora_do_mapa": F.col("classe").isNull(),
        "classe_x_subgrupo": mapa_sub[F.col("classe")] != F.col("subgrupo_tarifario"),
    })
    return df, {"linhas": total, **contar_flags(df)}


executar("unidades_consumidoras", construir_unidades_consumidoras)

## 2. faturas
`v_energy.bronze.faturas` → `v_energy.silver.faturas`. Regras FAT-01 a FAT-05. Depende de `unidades_consumidoras` (seção 1).

**Tratamentos**
- Cabeçalho repetido removido (FAT-01).
- `data_emissao` em 4 formatos convertida linha a linha (FAT-02); `data_vencimento` e `data_pagamento` como `date`.
- `valor_total` no formato `R$ 1.234,56` convertido (FAT-03); valores como `decimal(12,2)`.
- `data_pagamento` com ano `9999` (código de "não pago") vira nulo (FAT-04).

**Coluna derivada:** `status_pagamento` (`PAGA`, `PAGA COM ATRASO`, `EM ABERTO`), comparando `data_pagamento` com `data_vencimento`. O `status` original é mantido.

**Flag, sem apagar a linha**
- `flag_uc_orfa`: fatura de UC que não existe no cadastro (FAT-05). O destino final (quarentena) é decidido na integridade referencial.

**Validações antes de gravar:** `id_fatura` único, bandeira no domínio, `mes_referencia` em `AAAA-MM`, `status` coerente com `data_pagamento`, pagamento nunca antes da emissão, `valor_total = valor_energia + tributos`.

In [0]:
BANDEIRAS = ["Verde", "Amarela", "Vermelha P1", "Vermelha P2"]


def construir_faturas():
    df = spark.table(f"{BRONZE}.faturas")
    origem = [c for c in df.columns if c not in COLS_TECNICAS]

    df = df.filter(F.col("id_fatura").isNull() | (F.upper(F.trim("id_fatura")) != "ID_FATURA"))   # FAT-01 (chave nula não some aqui)
    df = limpar_todas(df, origem)
    df = df.withColumn("data_pagamento",                                       # FAT-04: 9999 = não pago
        F.when(F.col("data_pagamento").startswith("9999"), None).otherwise(F.col("data_pagamento")))

    convertidas = ["data_emissao", "data_vencimento", "data_pagamento", "valor_energia", "tributos", "valor_total"]
    df = guardar_texto(df, convertidas)
    vt = F.col("valor_total")
    df = (df
        .withColumn("data_emissao", data_multi("data_emissao"))                # FAT-02
        .withColumn("data_vencimento", data_multi("data_vencimento"))
        .withColumn("data_pagamento", data_multi("data_pagamento"))
        .withColumn("valor_total", F.when(vt.contains(","),                    # FAT-03: R$ 1.234,56
            F.regexp_replace(F.regexp_replace(F.regexp_replace(vt, r"R\$\s*", ""), r"\.", ""), ",", ".")).otherwise(vt)))
    for c in ["valor_energia", "tributos", "valor_total"]:
        df = df.withColumn(c, dec(c, 12, 2))
    df = exigir_sem_perda(df, convertidas)

    df = df.withColumn("status_pagamento",
        F.when(F.col("data_pagamento").isNull(), "EM ABERTO")
         .when(F.col("data_pagamento") > F.col("data_vencimento"), "PAGA COM ATRASO")
         .otherwise("PAGA"))

    ucs = obter("unidades_consumidoras").select("id_uc", F.lit(True).alias("_existe"))
    df = (df.join(ucs, "id_uc", "left")                                        # FAT-05
            .withColumn("flag_uc_orfa", F.col("_existe").isNull()).drop("_existe"))
    df = df.select(*origem, "status_pagamento", "flag_uc_orfa", *COLS_TECNICAS)

    exigir_zero(df, {"id_fatura_nulo": F.col("id_fatura").isNull()})
    total = exigir_chave_unica(df, ["id_fatura"])
    exigir_zero(df, {
        "bandeira_fora_do_dominio": ~F.col("bandeira_tarifaria").isin(BANDEIRAS),
        "mes_referencia_fora_do_formato": ~F.col("mes_referencia").rlike(r"^\d{4}-(0[1-9]|1[0-2])$"),
        "status_x_data_pagamento": (F.col("status") == "EM ABERTO") != (F.col("status_pagamento") == "EM ABERTO"),
        "pagamento_antes_da_emissao": F.col("data_pagamento") < F.col("data_emissao"),
        "total_diferente_de_energia_mais_tributos":
            F.abs(F.col("valor_total") - F.col("valor_energia") - F.col("tributos")) > 0.01,
    })
    return df, {"linhas": total, **contar_flags(df)}


executar("faturas", construir_faturas)

## 3. leituras
`v_energy.bronze.leituras` (5 arquivos anuais) → `v_energy.silver.leituras`. Regras LEI-01 a LEI-07. Depende de `unidades_consumidoras` (seção 1).

**Tratamentos**
- Cópias exatas removidas (LEI-01).
- `data_leitura` em 4 formatos convertida linha a linha (LEI-02); leituras e `consumo_kwh` como `decimal(14,2)`.
- `leitura_anterior` ausente (nulo, `N/A`, `NULL` ou `-1`) vira nulo (LEI-03).
- Mesma leitura registrada duas vezes na UC × competência (muda só `id_leitura` e `id_medidor`): fica a leitura cujo medidor pertence à UC no cadastro de medidores (LEI-04); se um par não tiver exatamente uma leitura assim, o notebook para. 1 leitura por UC × competência.

**Flags, sem apagar a linha.** `consumo_kwh` continua sendo o valor oficial; as flags indicam onde ele é suspeito.
- `flag_leitura_duplicada` (LEI-04), `flag_sem_leitura_anterior` (LEI-03), `flag_consumo_negativo` (LEI-05),
  `flag_leitura_regressiva` (LEI-06: atual menor que a anterior), `flag_consumo_divergente` (LEI-07: consumo difere de `atual − anterior` em mais de 5% ou 1 kWh, o que for maior).

**Validações antes de gravar:** `id_leitura` e UC × competência únicos, `tipo_leitura` no domínio, `mes_referencia` em `AAAA-MM`, nenhuma leitura de UC fora do cadastro.

In [0]:
def construir_leituras():
    df = spark.table(f"{BRONZE}.leituras")
    origem = [c for c in df.columns if c not in COLS_TECNICAS]

    df = limpar_todas(df, origem)                                              # espaços, N/A, NULL
    df = df.withColumn("leitura_anterior",                                     # LEI-03: -1 = sem leitura
        F.when(F.col("leitura_anterior") == "-1", None).otherwise(F.col("leitura_anterior")))
    df = df.dropDuplicates(origem)                                             # LEI-01

    convertidas = ["data_leitura", "leitura_anterior", "leitura_atual", "consumo_kwh"]
    df = guardar_texto(df, convertidas)
    df = df.withColumn("data_leitura", data_multi("data_leitura"))             # LEI-02
    for c in ["leitura_anterior", "leitura_atual", "consumo_kwh"]:
        df = df.withColumn(c, dec(c, 14, 2))
    df = exigir_sem_perda(df, convertidas)

    medidor_da_uc = (spark.table(f"{BRONZE}.medidores")                         # LEI-04: par (medidor, UC) do cadastro
        .select(limpar("id_medidor").alias("id_medidor"), limpar("id_uc").alias("id_uc"))
        .dropna().dropDuplicates().withColumn("_medidor_da_uc", F.lit(1)))
    df = df.join(medidor_da_uc, ["id_medidor", "id_uc"], "left").fillna(0, ["_medidor_da_uc"])
    w = Window.partitionBy("id_uc", "mes_referencia")
    df = (df.withColumn("_n", F.count(F.lit(1)).over(w))
            .withColumn("_do_medidor_da_uc", F.sum("_medidor_da_uc").over(w)))
    exigir_zero(df, {"duplicada_sem_exatamente_um_medidor_da_uc": (F.col("_n") > 1) & (F.col("_do_medidor_da_uc") != 1)})
    df = (df.withColumn("_ordem", F.row_number().over(w.orderBy(F.col("_medidor_da_uc").desc(), "id_leitura")))
            .withColumn("flag_leitura_duplicada", F.col("_n") > 1)
            .filter("_ordem = 1").drop("_n", "_ordem", "_medidor_da_uc", "_do_medidor_da_uc"))

    delta = F.col("leitura_atual") - F.col("leitura_anterior")
    df = (df
        .withColumn("flag_sem_leitura_anterior", F.col("leitura_anterior").isNull())             # LEI-03
        .withColumn("flag_consumo_negativo", F.col("consumo_kwh") < 0)                           # LEI-05
        .withColumn("flag_leitura_regressiva", F.coalesce(delta < 0, F.lit(False)))              # LEI-06
        .withColumn("flag_consumo_divergente", F.coalesce(                                       # LEI-07
            F.abs(F.col("consumo_kwh") - delta) > F.greatest(F.abs(delta) * 0.05, F.lit(1)), F.lit(False))))
    flags = [c for c in df.columns if c.startswith("flag_")]
    df = df.select(*origem, *flags, *COLS_TECNICAS)

    total = exigir_chave_unica(df, ["id_leitura"])
    assert df.select("id_uc", "mes_referencia").distinct().count() == total, "UC × competência duplicada"
    exigir_zero(df, {
        "tipo_leitura_fora_do_dominio": ~F.col("tipo_leitura").isin("REAL", "ESTIMADA"),
        "mes_referencia_fora_do_formato": ~F.col("mes_referencia").rlike(r"^\d{4}-(0[1-9]|1[0-2])$"),
    })
    exigir_sem_orfaos(df, "leituras")
    return df, {"linhas": total, **contar_flags(df)}


executar("leituras", construir_leituras)

## 4. medidores
`v_energy.bronze.medidores` → `v_energy.silver.medidores`. Regras MED-01 a MED-06.

**Tratamentos**
- Cópias exatas removidas (MED-01): `id_medidor` único.
- Espaços removidos e `modelo` em maiúsculas (MED-02); `tipo` padronizado para `ELETRONICO`, `ELETROMECANICO` e `SMART` (MED-03).
- `data_instalacao` em 4 formatos convertida linha a linha (MED-04); `data_retirada` com `9999-12-31` (medidor ativo) vira nulo (MED-05).
- `data_ultima_manutencao` vem vazia na origem e fica como `date` nula (MED-06).

**Validações antes de gravar:** `id_medidor` único, `tipo` e `status` no domínio, `ATIVO` sem `data_retirada` e `RETIRADO` com `data_retirada` não anterior à instalação, no máximo 1 medidor `ATIVO` por UC, nenhum medidor de UC fora do cadastro.

In [0]:
def construir_medidores():
    df = spark.table(f"{BRONZE}.medidores")
    origem = [c for c in df.columns if c not in COLS_TECNICAS]

    df = limpar_todas(df, origem)                                              # MED-02 (espaços)
    df = df.dropDuplicates(origem)                                             # MED-01 (cópias exatas, já sem espaços)
    df = df.withColumn("data_retirada",                                        # MED-05
        F.when(F.col("data_retirada") == "9999-12-31", None).otherwise(F.col("data_retirada")))

    tipo = sem_acento("tipo")
    df = (df
        .withColumn("modelo", F.upper("modelo"))
        .withColumn("tipo", F.when(tipo.contains("SMART"), "SMART")            # MED-03
                             .when(tipo.contains("MECANICO"), "ELETROMECANICO")
                             .when(tipo.contains("ELETRONICO"), "ELETRONICO")
                             .otherwise(tipo))
        .withColumn("status", F.upper("status")))

    datas = ["data_instalacao", "data_retirada", "data_ultima_manutencao"]
    df = guardar_texto(df, datas)
    for c in datas:
        df = df.withColumn(c, data_multi(c))                                   # MED-04 e MED-06
    df = exigir_sem_perda(df, datas)
    df = df.select(*origem, *COLS_TECNICAS)

    total = exigir_chave_unica(df, ["id_medidor"])
    exigir_zero(df, {
        "tipo_fora_do_dominio": ~F.col("tipo").isin("ELETRONICO", "ELETROMECANICO", "SMART"),
        "status_fora_do_dominio": ~F.col("status").isin("ATIVO", "RETIRADO"),
        "ativo_com_retirada": (F.col("status") == "ATIVO") & F.col("data_retirada").isNotNull(),
        "retirado_sem_retirada": (F.col("status") == "RETIRADO") & F.col("data_retirada").isNull(),
        "retirada_antes_da_instalacao": F.col("data_retirada") < F.col("data_instalacao"),
    })
    ativos_por_uc = df.filter(F.col("status") == "ATIVO").groupBy("id_uc").count().filter("count > 1").count()
    assert ativos_por_uc == 0, f"{ativos_por_uc:,} UCs com mais de 1 medidor ATIVO"
    exigir_sem_orfaos(df, "medidores")
    return df, {"linhas": total}


executar("medidores", construir_medidores)

## 5. geracao_distribuida
`v_energy.bronze.geracao_distribuida` → `v_energy.silver.geracao_distribuida`. Regras GD-01 e GD-02.

**Tratamentos**
- `potencia_kwp` com vírgula decimal convertida para `decimal` (GD-01); `energia_injetada_kwh_media_mensal` como `decimal`; `data_conexao` como `date`; `modalidade` em maiúsculas.

**Flag, sem apagar a linha**
- `flag_potencia_anomala`: potência acima de 500 kWp, compatível com registro em Wp em vez de kWp (GD-02).

**Validações antes de gravar:** 1 linha por `id_uc`, nenhuma conversão perdida, nenhuma GD de UC fora do cadastro.

In [0]:
def construir_geracao_distribuida():
    df = spark.table(f"{BRONZE}.geracao_distribuida")
    origem = [c for c in df.columns if c not in COLS_TECNICAS]

    df = limpar_todas(df, origem)
    df = df.dropDuplicates(origem)                                             # depois de limpar: espaço não esconde cópia
    convertidas = ["potencia_kwp", "energia_injetada_kwh_media_mensal", "data_conexao"]
    df = guardar_texto(df, convertidas)
    df = (df
        .withColumn("potencia_kwp", dec("potencia_kwp", 12, 2))                             # GD-01
        .withColumn("energia_injetada_kwh_media_mensal", dec("energia_injetada_kwh_media_mensal", 14, 2))
        .withColumn("data_conexao", data_multi("data_conexao"))
        .withColumn("modalidade", F.upper("modalidade")))
    df = exigir_sem_perda(df, convertidas)
    df = df.withColumn("flag_potencia_anomala", F.col("potencia_kwp") > 500)                # GD-02
    df = df.select(*origem, "flag_potencia_anomala", *COLS_TECNICAS)

    total = exigir_chave_unica(df, ["id_uc"])
    exigir_sem_orfaos(df, "geracao_distribuida")
    return df, {"linhas": total, **contar_flags(df)}


executar("geracao_distribuida", construir_geracao_distribuida)

## 6. ocorrencias_rede
`v_energy.bronze.ocorrencias_rede` → `v_energy.silver.ocorrencias_rede`. Regras OCO-01 a OCO-03.

**Tratamentos**
- `causa` ausente (vazio, `-`, `N/A`, `NULL`) vira nulo (OCO-01); `tipo`, `causa` e `alimentador` em maiúsculas.
- `duracao_min` sem o sufixo `" min"`, como `decimal` (OCO-02).
- `data_abertura` e `data_encerramento` como `timestamp` (OCO-03).

**Flag, sem apagar a linha**
- `flag_causa_ausente` (OCO-01).

**Validações antes de gravar:** `id_ocorrencia` único, encerramento depois da abertura, `duracao_min` igual à diferença entre encerramento e abertura, nenhuma ocorrência de UC fora do cadastro.

In [0]:
def construir_ocorrencias_rede():
    df = spark.table(f"{BRONZE}.ocorrencias_rede")
    origem = [c for c in df.columns if c not in COLS_TECNICAS]

    df = limpar_todas(df, origem)                                              # OCO-01
    df = df.dropDuplicates(origem)                                             # depois de limpar: espaço não esconde cópia
    df = (df
        .withColumn("alimentador", F.upper("alimentador"))
        .withColumn("tipo", F.upper("tipo"))
        .withColumn("causa", F.upper("causa"))
        .withColumn("flag_causa_ausente", F.col("causa").isNull())
        .withColumn("duracao_min", F.regexp_replace("duracao_min", r"\s*min$", "")))       # OCO-02

    convertidas = ["data_abertura", "data_encerramento", "duracao_min"]
    df = guardar_texto(df, convertidas)
    df = (df
        .withColumn("data_abertura", ts("data_abertura"))                                    # OCO-03
        .withColumn("data_encerramento", ts("data_encerramento"))
        .withColumn("duracao_min", dec("duracao_min", 12, 2)))
    df = exigir_sem_perda(df, convertidas)
    df = df.select(*origem, "flag_causa_ausente", *COLS_TECNICAS)

    total = exigir_chave_unica(df, ["id_ocorrencia"])
    minutos = (F.unix_timestamp("data_encerramento") - F.unix_timestamp("data_abertura")) / 60
    exigir_zero(df, {
        "encerramento_antes_da_abertura": F.col("data_encerramento") <= F.col("data_abertura"),
        "duracao_diferente_das_datas": F.abs(F.col("duracao_min") - minutos) > 0.01,
    })
    exigir_sem_orfaos(df, "ocorrencias_rede")
    return df, {"linhas": total, **contar_flags(df)}


executar("ocorrencias_rede", construir_ocorrencias_rede)

## 7. atendimentos
`v_energy.bronze.atendimentos` (JSON aninhado) → `v_energy.silver.atendimentos`. Regras ATD-01 a ATD-03.

**Tratamentos**
- `canal`, `assunto` e `subassunto` em maiúsculas; `nota_satisfacao` como `decimal(4,2)` (ATD-01).
- `historico` mantém a estrutura (array de structs), com os campos tipados: `data` como `timestamp` e `sequencia` como inteiro (ATD-02).
- `data_abertura` e `data_fechamento` como `timestamp` (ATD-03).

**Flag, sem apagar a linha**
- `flag_sem_nota_satisfacao` (ATD-01).

**Validações antes de gravar:** colunas da Bronze iguais às esperadas, `protocolo` único, fechamento depois da abertura, nota entre 1 e 5, nenhuma data ou sequência do histórico perdida na conversão, nenhum atendimento de UC fora do cadastro.

In [0]:
COLUNAS_ATENDIMENTO = ["protocolo", "id_uc", "canal", "assunto", "subassunto",
                       "data_abertura", "data_fechamento", "nota_satisfacao", "historico"]
HISTORICO_TIPADO = """transform(historico, x -> named_struct(
    'acao', upper(trim(x.acao)),
    'canal', upper(trim(x.canal)),
    'data', try_to_timestamp(replace(trim(x.data), 'T', ' '), 'yyyy-MM-dd HH:mm:ss'),
    'sequencia', try_cast(trim(x.sequencia) AS INT)))"""


def nao_nulos(array, campo):
    return F.expr(f"size(filter({array}, x -> x.{campo} IS NOT NULL))")


def construir_atendimentos():
    df = spark.table(f"{BRONZE}.atendimentos")
    exigir_colunas(df, COLUNAS_ATENDIMENTO)
    texto = [c for c in COLUNAS_ATENDIMENTO if c != "historico"]

    df = limpar_todas(df, texto)
    df = (df
        .withColumn("canal", F.upper("canal"))
        .withColumn("assunto", F.upper("assunto"))
        .withColumn("subassunto", F.upper("subassunto")))

    convertidas = ["data_abertura", "data_fechamento", "nota_satisfacao"]
    df = guardar_texto(df, convertidas).withColumn("_historico_origem", F.col("historico"))
    df = (df
        .withColumn("data_abertura", ts("data_abertura"))                                    # ATD-03
        .withColumn("data_fechamento", ts("data_fechamento"))
        .withColumn("nota_satisfacao", dec("nota_satisfacao", 4, 2))                         # ATD-01
        .withColumn("historico", F.expr(HISTORICO_TIPADO)))                                  # ATD-02
    df = exigir_sem_perda(df, convertidas)
    exigir_zero(df, {
        "historico_data_perdida": nao_nulos("_historico_origem", "data") != nao_nulos("historico", "data"),
        "historico_sequencia_perdida": nao_nulos("_historico_origem", "sequencia") != nao_nulos("historico", "sequencia"),
    })
    df = df.withColumn("flag_sem_nota_satisfacao", F.col("nota_satisfacao").isNull())       # ATD-01
    df = df.select(*COLUNAS_ATENDIMENTO, "flag_sem_nota_satisfacao", *COLS_TECNICAS)

    total = exigir_chave_unica(df, ["protocolo"])
    exigir_zero(df, {
        "fechamento_antes_da_abertura": F.col("data_fechamento") <= F.col("data_abertura"),
        "nota_fora_de_1_a_5": (F.col("nota_satisfacao") < 1) | (F.col("nota_satisfacao") > 5),
    })
    exigir_sem_orfaos(df, "atendimentos")
    return df, {"linhas": total, **contar_flags(df)}


executar("atendimentos", construir_atendimentos)

## 8. inspecoes (3 abas → 1 tabela)
`v_energy.bronze.inspecoes_ago_set`, `inspecoes_out_nov` e `inspecoes_dez` → `v_energy.silver.inspecoes` (uma tabela só; a aba de origem fica na coluna `aba_origem` e é usada na flag INSP-05). Regras INSP-01 a INSP-07.

**Tratamentos**
- `resultado` padronizado para `IRREGULARIDADE CONFIRMADA` / `SEM IRREGULARIDADE` (INSP-01); `NADA CONSTA` tratado como `SEM IRREGULARIDADE` (INSP-02).
- `tipo_irregularidade` ausente (vazio, `-`, `N/A`, `NULL`) vira nulo (INSP-03).
- `data` em 4 formatos convertida linha a linha (INSP-04); `inspetor` sem espaços sobrando (INSP-06); `kwh_recuperado` com vírgula decimal convertido (INSP-07).

**Flags, sem apagar a linha**
- `flag_resultado_nada_consta`: o resultado de origem era `NADA CONSTA` (INSP-02).
- `flag_aba_divergente`: a data da inspeção não pertence aos meses da aba (INSP-05). Vale a `data`.

**Validações antes de gravar:** `id_inspecao` único, 1 inspeção por UC, `resultado` no domínio, coerência resultado × tipo × kWh (confirmada tem tipo e kWh > 0; sem irregularidade não tem tipo e tem kWh zero), nenhuma inspeção de UC fora do cadastro.

In [0]:
ABAS_INSPECOES = {"inspecoes_ago_set": [8, 9], "inspecoes_out_nov": [10, 11], "inspecoes_dez": [12]}
COLUNAS_INSPECAO = ["id_inspecao", "id_uc", "data", "inspetor", "resultado", "tipo_irregularidade", "kwh_recuperado"]


def tratar_aba(tabela, meses):
    df = spark.table(f"{BRONZE}.{tabela}")
    exigir_colunas(df, COLUNAS_INSPECAO)
    df = limpar_todas(df, COLUNAS_INSPECAO)                                    # INSP-03 e INSP-06

    resultado = F.upper("resultado")
    df = (df
        .withColumn("flag_resultado_nada_consta", F.coalesce(resultado == "NADA CONSTA", F.lit(False)))   # INSP-02
        .withColumn("resultado",                                                                         # INSP-01
            F.when(resultado.isin("IRREGULARIDADE CONFIRMADA", "CONFIRMADA"), "IRREGULARIDADE CONFIRMADA")
             .when(resultado.isin("SEM IRREGULARIDADE", "NADA CONSTA"), "SEM IRREGULARIDADE")
             .otherwise(resultado)))

    df = guardar_texto(df, ["data", "kwh_recuperado"])
    df = (df
        .withColumn("data", data_multi("data"))                                # INSP-04
        .withColumn("kwh_recuperado", dec("kwh_recuperado", 14, 2)))           # INSP-07
    df = exigir_sem_perda(df, ["data", "kwh_recuperado"])
    df = df.withColumn("flag_aba_divergente", F.coalesce(~F.month("data").isin(meses), F.lit(False)))   # INSP-05
    df = df.withColumn("aba_origem", F.lit(tabela))                            # de qual aba veio a linha
    return df.select(*COLUNAS_INSPECAO, "aba_origem", "flag_resultado_nada_consta", "flag_aba_divergente", *COLS_TECNICAS)


def construir_inspecoes():
    abas = [tratar_aba(tabela, meses) for tabela, meses in ABAS_INSPECOES.items()]
    df = abas[0]
    for outra in abas[1:]:
        df = df.unionByName(outra)

    total = exigir_chave_unica(df, ["id_inspecao"])
    assert df.select("id_uc").distinct().count() == total, "UC com mais de uma inspeção"
    confirmada = F.col("resultado") == "IRREGULARIDADE CONFIRMADA"
    tem_tipo = F.col("tipo_irregularidade").isNotNull()
    exigir_zero(df, {
        "resultado_fora_do_dominio": ~F.col("resultado").isin("IRREGULARIDADE CONFIRMADA", "SEM IRREGULARIDADE"),
        "confirmada_sem_tipo_ou_kwh": confirmada & (~tem_tipo | ~(F.col("kwh_recuperado") > 0)),
        "sem_irregularidade_com_tipo_ou_kwh": ~confirmada & (tem_tipo | (F.col("kwh_recuperado") != 0)),
    })
    exigir_sem_orfaos(df, "inspecoes")
    return df, {"linhas": total, **contar_flags(df)}


executar("inspecoes", construir_inspecoes)

## 9. Validação cruzada
Conferências **entre** as tabelas Silver (descritas em `docs/qualidade-dados.md`). Usa as tabelas tratadas nesta execução ou, se a seção não rodou agora, as tabelas já gravadas. Não grava nada.

- **Pareamento:** cada fatura de UC válida tem exatamente uma leitura na mesma UC e competência, e vice-versa.
- **Encadeamento:** `leitura_anterior` de um mês = `leitura_atual` do mês anterior, na mesma UC.
- **Datas:** leitura dentro do mês da competência; fatura emitida depois da leitura.
- **Tarifa implícita** (`valor_energia ÷ consumo_kwh`) por subgrupo e ano: informativa, deve ficar em faixas estreitas.
- **Alimentadores:** alimentadores das ocorrências que não existem no cadastro de UCs (informativo).

In [0]:
def validar_cruzado():
    print("=== validacao_cruzada")
    try:
        f = obter("faturas").filter(~F.col("flag_uc_orfa")).select("id_uc", "mes_referencia", "id_fatura", "valor_energia", "data_emissao")
        l = obter("leituras").select("id_uc", "mes_referencia", "id_leitura", "consumo_kwh", "data_leitura",
                                     "leitura_anterior", "leitura_atual", "flag_consumo_negativo", "flag_consumo_divergente")
        u = obter("unidades_consumidoras").select("id_uc", "subgrupo_tarifario", "alimentador")
        j = f.join(l, ["id_uc", "mes_referencia"], "full_outer")

        w = Window.partitionBy("id_uc").orderBy("mes_referencia")
        e = l.withColumn("atual_mes_passado", F.lag("leitura_atual").over(w))
        exigir_zero(j, {
            "fatura_sem_leitura": F.col("id_leitura").isNull(),
            "leitura_sem_fatura": F.col("id_fatura").isNull(),
            "fatura_emitida_antes_da_leitura": F.col("data_emissao") < F.col("data_leitura"),
        })
        exigir_zero(e, {
            "encadeamento_quebrado": F.col("leitura_anterior") != F.col("atual_mes_passado"),
            "leitura_fora_do_mes": F.date_format("data_leitura", "yyyy-MM") != F.col("mes_referencia"),
        })

        tarifa = (j.filter(F.col("consumo_kwh") > 0)
                   .filter(~F.col("flag_consumo_negativo") & ~F.col("flag_consumo_divergente"))
                   .join(u, "id_uc")
                   .withColumn("tarifa", F.col("valor_energia") / F.col("consumo_kwh"))
                   .groupBy("subgrupo_tarifario", F.substring("mes_referencia", 1, 4).alias("ano"))
                   .agg(F.expr("percentile_approx(tarifa, array(0.05, 0.5, 0.95))").alias("p5_mediana_p95"))
                   .orderBy("subgrupo_tarifario", "ano"))
        tarifa.show(30, truncate=False)

        alimentadores_fora = (obter("ocorrencias_rede").select("alimentador").distinct()
                              .join(u.select("alimentador").distinct(), "alimentador", "left_anti").count())
        print("alimentadores das ocorrências fora do cadastro de UCs:", alimentadores_fora)

        RESULTADOS["validacao_cruzada"] = {"status": "conferida", "alimentadores_fora_do_cadastro": alimentadores_fora}
        print("[OK] validacao_cruzada: conferida")
    except Exception as e:
        RESULTADOS["validacao_cruzada"] = {"status": "FALHA", "erro": f"{type(e).__name__}: {e}"}
        print("[FALHA] validacao_cruzada")
        traceback.print_exc()


validar_cruzado()

## 10. Resumo
Imprime o estado de cada tabela. Output **confidencial** (contagens reais): nunca no Git.
Falha se alguma tabela não foi executada ou falhou.

In [0]:
print("CONFIDENCIAL: resultados derivados do dataset. Não versionar.")
print(f"gravar = {'sim' if GRAVAR else 'nao'} | {datetime.now().isoformat(timespec='seconds')}")
esperadas = TABELAS_ESPERADAS + (["validacao_cruzada"] if VALIDACAO_ESPERADA else [])
incompletas = []
for nome in esperadas:
    r = RESULTADOS.get(nome)
    if r is None:
        print(f"- {nome}: NÃO EXECUTADA")
        incompletas.append(nome)
        continue
    print(f"- {nome}: {r}")
    if r["status"] == "FALHA":
        incompletas.append(nome)

assert not incompletas, f"Silver NÃO concluída. Não executadas ou com falha: {incompletas}"
print("[OK] Todas as tabelas tratadas e validadas" + (" e gravadas." if GRAVAR else " (nada foi gravado: gravar = nao)."))